# Wine Buddy

Wine analysis project: data scraping, similarity search, recommendations, and food pairing.

> **No pre-scraped data is included.** You must run the scraper yourself.

# 1. Data Gathering

In [ ]:
import numpy as np
import requests
import pandas as pd
import json
import re

import seaborn as sns
from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"] = (20, 8)

## Scrape wines

Scrape wine data from Vivino's public API.
Please respect their Terms of Service and rate-limit your requests.

In [ ]:
def get_wine_data(wine_id, year, page):
    headers = {"User-Agent": "wine-buddy/0.1"}
    api_url = "https://www.vivino.com/api/wines/{id}/reviews?per_page=50&year={year}&page={page}"
    data = requests.get(
        api_url.format(id=wine_id, year=year, page=page), headers=headers
    ).json()
    return data

In [ ]:
column_names = [
    "id", "year", "wine_id", "wine_seo_name", "winery_seo_name",
    "is_natural", "structure_acidity", "structure_fizziness",
    "structure_intensity", "structure_sweetness", "structure_tannin",
    "flavor", "ratings_count", "ratings_average",
    "style_body", "style_acidity", "style_food", "style_grapes",
    "region_seo_name", "region_country_code",
]

# Configure your scraping parameters
COUNTRY_CODE = "IT"
WINE_TYPES = {
    "1": "red", "2": "white", "3": "sparkling",
    "4": "pink", "7": "dessert", "24": "liquor",
}

In [ ]:
def scrape_wines(country_code, wine_type_id, wine_type_name, min_price=5, max_price=50):
    """Scrape wine listings from Vivino explore API."""
    results = []
    distinct_ids = set()
    n_ids = 0

    for p in range(1, 100):
        r = requests.get(
            "https://www.vivino.com/api/explore/explore",
            params={
                "country_code": country_code,
                "country_codes[]": country_code.lower(),
                "currency_code": "EUR",
                "grape_filter": "varietal",
                "min_rating": "1",
                "order_by": "price",
                "order": "asc",
                "page": p,
                "price_range_max": str(max_price),
                "price_range_min": str(min_price),
                "wine_type_ids[]": wine_type_id,
            },
            headers={"User-Agent": "wine-buddy/0.1"},
        )

        wines = r.json()["explore_vintage"]["matches"]
        if not wines:
            break
        if p % 25 == 0:
            print(f"Page {p} - {len(wines)} retrieved")

        for t in wines:
            try:
                results.append((
                    t["vintage"]["id"],
                    t["vintage"]["year"],
                    t["vintage"]["wine"]["id"],
                    t["vintage"]["wine"]["seo_name"],
                    t["vintage"]["wine"]["winery"]["seo_name"],
                    t["vintage"]["wine"]["is_natural"],
                    t["vintage"]["wine"]["taste"]["structure"]["acidity"],
                    t["vintage"]["wine"]["taste"]["structure"]["fizziness"],
                    t["vintage"]["wine"]["taste"]["structure"]["intensity"],
                    t["vintage"]["wine"]["taste"]["structure"]["sweetness"],
                    t["vintage"]["wine"]["taste"]["structure"]["tannin"],
                    t["vintage"]["wine"]["taste"]["flavor"],
                    t["vintage"]["wine"]["statistics"]["ratings_count"],
                    t["vintage"]["wine"]["statistics"]["ratings_average"],
                    t["vintage"]["wine"]["style"]["body"],
                    t["vintage"]["wine"]["style"]["acidity"],
                    t["vintage"]["wine"]["style"]["food"],
                    t["vintage"]["wine"]["style"]["grapes"],
                    t["vintage"]["wine"]["region"]["seo_name"],
                    t["vintage"]["wine"]["region"]["country"]["code"],
                ))
                distinct_ids.add(t["vintage"]["id"])
            except (KeyError, TypeError):
                pass

        if len(distinct_ids) <= n_ids:
            print("Reached max number of wines")
            break
        n_ids = len(distinct_ids)

    df = pd.DataFrame(results, columns=column_names)
    df["wine_type"] = wine_type_name
    return df.groupby("id").first().reset_index()

In [ ]:
# Uncomment to run the scraper (this will take a while)
# import os
# os.makedirs("data", exist_ok=True)
#
# for type_id, type_name in WINE_TYPES.items():
#     print(f"Scraping {type_name} wines...")
#     df = scrape_wines(COUNTRY_CODE, type_id, type_name)
#     df.to_csv(f"data/{COUNTRY_CODE.lower()}_{type_name}.csv", index=False)
#     print(f"  Saved {len(df)} {type_name} wines")

# 2. Data Preparation

## Parse JSON columns

The scraped data contains nested JSON strings for flavors, grapes, and food pairings.
These need to be parsed and pivoted into usable features.

### Flavor extraction

In [ ]:
def get_flavors(color="red"):
    df = pd.read_csv(f"data/{COUNTRY_CODE.lower()}_{color}.csv")

    def parse_flavor(f_str):
        try:
            if isinstance(f_str, str):
                f_clean = re.sub(r"(?<=[A-z])'(?=[A-z])", "", f_str)
                f_clean = re.sub(r"'", '"', f_clean)
                f_obj = json.loads(f_clean)
                return [
                    {"flavor": o["group"], "flavor_count": o["stats"]["count"], "flavor_score": o["stats"]["score"]}
                    for o in f_obj
                ]
        except Exception as e:
            print(e)
        return {"flavor": None, "flavor_count": None, "flavor_score": None}

    df["flavor"] = df["flavor"].apply(parse_flavor)
    df = df.explode(column="flavor")
    df = (
        pd.concat([df.drop(["flavor"], axis=1), df["flavor"].apply(pd.Series)], axis=1)
        .drop(0, axis=1)
        .drop_duplicates()
    )
    df = df[df["flavor"].notna()]
    return (
        pd.pivot(df, index="id", columns="flavor", values="flavor_score")
        .fillna(0.0)
        .reset_index()
        .rename_axis(None, axis=1)
    )

### Grape extraction

In [ ]:
def get_grapes(color="red"):
    df = pd.read_csv(f"data/{COUNTRY_CODE.lower()}_{color}.csv")

    def parse_grapes(f_str):
        try:
            if isinstance(f_str, str):
                f_clean = re.sub(r"(?<=[A-z])'(?=[A-z])", "", f_str)
                f_clean = re.sub(r"'", '"', f_clean)
                f_clean = re.sub("True", "true", f_clean)
                f_clean = re.sub("False", "false", f_clean)
                f_obj = json.loads(f_clean)
                return [{"grapes": o["seo_name"]} for o in f_obj]
        except Exception as e:
            print(e, f_str)
        return {"grapes": None}

    df["grapes"] = df["style_grapes"].apply(parse_grapes)
    df = df.explode(column="grapes")
    df = (
        pd.concat([df.drop(["grapes", "style_grapes"], axis=1), df["grapes"].apply(pd.Series)], axis=1)
        .drop_duplicates()
    )
    df = df[df["grapes"].notna()]
    return (
        pd.get_dummies(df[["id", "grapes"]], columns=["grapes"])
        .groupby("id")
        .sum()
        .reset_index()
        .rename_axis(None, axis=1)
    )

### Food pairing extraction

In [ ]:
def get_food(df):
    red = df.copy()

    def parse_food(f_str):
        try:
            if isinstance(f_str, str):
                f_clean = re.sub(r"(?<=[A-z])'(?=[A-z])", "", f_str)
                f_clean = re.sub(r"'", '"', f_clean)
                f_clean = re.sub("True", "true", f_clean)
                f_clean = re.sub("False", "false", f_clean)
                f_clean = re.sub("None", "null", f_clean)
                f_obj = json.loads(f_clean)
                return [{"style_food": o["seo_name"]} for o in f_obj]
        except Exception as e:
            print(e, f_str)
        return {"style_food": None}

    red["food"] = red["style_food"].apply(parse_food)
    red = red.explode(column="food")
    red = (
        pd.concat([red.drop(["food", "style_food"], axis=1), red["food"].apply(pd.Series)], axis=1)
        .drop_duplicates()
    )
    red = red[red["style_food"].notna()]
    return (
        pd.get_dummies(red[["id", "style_food"]], columns=["style_food"], prefix="", prefix_sep="")
        .groupby("id")
        .sum()
        .reset_index()
        .rename_axis(None, axis=1)
    )

### Flavour score pivot

In [ ]:
def get_flavour(df):
    red = df.copy()

    def parse_flavor(f_str):
        try:
            if isinstance(f_str, str):
                f_clean = re.sub(r"(?<=[A-z])'(?=[A-z])", "", f_str)
                f_clean = re.sub(r"'", '"', f_clean)
                f_clean = re.sub("True", "true", f_clean)
                f_clean = re.sub("False", "false", f_clean)
                f_clean = re.sub("None", "null", f_clean)
                f_obj = json.loads(f_clean)
                return [(o["group"], o["stats"]["score"]) for o in f_obj]
        except Exception as e:
            print(e, f_str)
        return {"flavor": None}

    red["flavor_desc"] = red["flavor"].apply(parse_flavor)
    red = red.explode(column="flavor_desc")
    red["flavor_name"] = red["flavor_desc"].map(lambda x: x[0])
    red["flavor_score"] = red["flavor_desc"].apply(lambda x: x[1])
    red = red[red["flavor"].notna()]
    return (
        pd.pivot_table(data=red, values="flavor_score", index="id", columns="flavor_name")
        .reset_index()
        .rename_axis(None, axis=1)
        .fillna(0)
    )

## Merge wines and clean data

In [ ]:
# Load your scraped data files
reds = pd.read_csv(f"data/{COUNTRY_CODE.lower()}_red.csv")
white = pd.read_csv(f"data/{COUNTRY_CODE.lower()}_white.csv")
spark = pd.read_csv(f"data/{COUNTRY_CODE.lower()}_sparkling.csv")

wines = pd.concat([reds, white, spark]).reset_index()
wines = pd.merge(wines, get_flavour(wines), on="id")
wines = wines.drop(["flavor", "style_food", "style_grapes", "index"], axis=1)

wines.head(3)

### Null values

Handle missing values for region, year, and wine structure attributes.

In [ ]:
wines.isnull().sum()

In [ ]:
# Fill missing region names (inspect and fill manually based on wine identity)
# wines.loc[idx, "region_seo_name"] = "region-name"

# Handle non-vintage years
wines["year"] = wines.year.replace("N.V.", np.nan)
wines["year"] = wines.year.fillna(wines["year"].astype("float").mean().astype("int"))

#### Sweetness, tannin and fizziness

- In white and sparkling wines the tannins are null — fill with **0** (tannins are negligible)
- Fizziness: fill with **0** for red and white (non-sparkling)
- Sweetness: Italian sparkling wines range widely (dry prosecco ~3 g/L to sweet asti spumante 50+ g/L) — fill with **average** for now

References:
- [Wine Folly - Prosecco Guide](https://winefolly.com/deep-dive/the-prosecco-wine-guide/)
- [Handy Wine Guide - Sweetness Chart](https://handywineguide.com/wine-sweetness-chart/)

In [ ]:
wines = wines.fillna({
    "structure_tannin": 0,
    "structure_fizziness": 0,
    "structure_sweetness": wines["structure_sweetness"].mean(),
})

# Drop is_natural if it has no variance
if wines["is_natural"].nunique() <= 1:
    wines = wines.drop("is_natural", axis=1)

### Distributions

In [ ]:
wines.describe()

In [ ]:
sns.heatmap(wines.corr(numeric_only=True));

In [ ]:
for c in wines.select_dtypes(include="number").columns.tolist():
    if c not in ["id", "wine_id"]:
        print(c)
        sns.displot(data=wines[c])
        plt.show()
        print()

### Generate embeddings

In [ ]:
ohe_cols = ["winery_seo_name", "region_seo_name"]

df = wines.copy()
for c in ohe_cols:
    one_hot = pd.get_dummies(df[c])
    df = df.drop(c, axis=1)
    df = df.join(one_hot)

ids = ["id", "wine_id", "wine_seo_name", "region_country_code"]
label = "wine_type"
features = [c for c in df.columns if c not in ids + [label]]

#### Dimensionality reduction comparison (PCA, LDA, NCA)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

n_neighbors = 3
random_state = 0
X, y = df[features], df[label].replace({"red": 1, "sparkling": 2, "white": 3})
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=random_state
)

dim_reduction_methods = [
    ("PCA", make_pipeline(StandardScaler(), PCA(n_components=2, random_state=random_state))),
    ("LDA", make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(n_components=2))),
    ("NCA", make_pipeline(StandardScaler(), NeighborhoodComponentsAnalysis(n_components=2, random_state=random_state))),
]

knn = KNeighborsClassifier(n_neighbors=n_neighbors)

for name, model in dim_reduction_methods:
    plt.figure()
    model.fit(X_train, y_train)
    knn.fit(model.transform(X_train), y_train)
    acc_knn = knn.score(model.transform(X_test), y_test)
    X_embedded = model.transform(X)
    sns.scatterplot(x=X_embedded[:, 0], y=X_embedded[:, 1], hue=df[label])
    plt.title(f"{name}, KNN (k={n_neighbors})\nTest accuracy = {acc_knn:.2f}")
plt.show()

# 3. Find Similar Wines

Use a KD-Tree on the embedded wine features to find nearest neighbors.

In [ ]:
from sklearn.neighbors import KDTree

tree = KDTree(X_embedded)

In [ ]:
# Search for a wine by name
wines[df["wine_seo_name"].str.contains("amarone", na=False)]

In [ ]:
# Query similar wines
query_id = 165052779  # replace with a wine ID from your data
query_wine = X_embedded[df["id"] == query_id]

dist, ind = tree.query(query_wine, k=5)
res = wines.iloc[ind[0]].copy()
res["dist"] = dist[0]
res.sort_values(by="dist")

# 4. Wine Recommendations

## Scrape user ratings

Collect individual user ratings for collaborative filtering.

Reference: [StackOverflow](https://stackoverflow.com/a/69288010/7306659)

In [ ]:
def scrape_ratings(wine_df):
    """Scrape individual wine ratings. Saves per-wine CSV files."""
    import os
    os.makedirs("data/ratings", exist_ok=True)

    for _, row in wine_df.iterrows():
        ratings = []
        print(f'Getting info about wine {row["wine_id"]}-{row["year"]}')
        page = 1
        while True:
            d = get_wine_data(row["wine_id"], row["year"], page)
            if not d["reviews"]:
                break
            for r in d["reviews"]:
                ratings.append([
                    row.get("year"),
                    row.get("wine_id"),
                    r.get("user", {}).get("id"),
                    r.get("statistics", {}).get("followers_count"),
                    r.get("statistics", {}).get("followings_count"),
                    r.get("statistics", {}).get("ratings_count"),
                    r.get("statistics", {}).get("ratings_sum"),
                    r.get("statistics", {}).get("reviews_count"),
                    r.get("statistics", {}).get("purchase_order_count"),
                    r.get("rating"),
                    r.get("note"),
                    r.get("created_at"),
                ])
            page += 1

        ratings_df = pd.DataFrame(ratings, columns=[
            "year", "wine_id", "customer_id", "followers_count", "followings_count",
            "ratings_count", "ratings_sum", "reviews_count", "purchase_order_count",
            "rating", "note", "created_at",
        ])
        ratings_df.to_csv(f'data/ratings/{COUNTRY_CODE.lower()}_{row["wine_id"]}.csv', index=False)

# Uncomment to run:
# scrape_ratings(df)

# 5. Wine & Food Pairing

Rule-based wine-food pairing engine.

Inspired by [RoaldSchuring/wine_food_pairing](https://github.com/RoaldSchuring/wine_food_pairing).

## Taste weight mappings

In [ ]:
food_weights = {
    "weight":  {1: (0, 0.3),  2: (0.3, 0.5),  3: (0.5, 0.7),  4: (0.7, 1)},
    "sweet":   {1: (0, 0.45), 2: (0.45, 0.6), 3: (0.6, 0.8),  4: (0.8, 1)},
    "acid":    {1: (0, 0.4),  2: (0.4, 0.55), 3: (0.55, 0.7), 4: (0.7, 1)},
    "salt":    {1: (0, 0.3),  2: (0.3, 0.55), 3: (0.55, 0.8), 4: (0.8, 1)},
    "piquant": {1: (0, 0.4),  2: (0.4, 0.6),  3: (0.6, 0.8),  4: (0.8, 1)},
    "fat":     {1: (0, 0.4),  2: (0.4, 0.5),  3: (0.5, 0.6),  4: (0.6, 1)},
    "bitter":  {1: (0, 0.3),  2: (0.3, 0.5),  3: (0.5, 0.65), 4: (0.65, 1)},
}

wine_weights = {
    "weight":  {1: (0, 0.25), 2: (0.25, 0.45), 3: (0.45, 0.75), 4: (0.75, 1)},
    "sweet":   {1: (0, 0.25), 2: (0.25, 0.6),  3: (0.6, 0.75),  4: (0.75, 1)},
    "acid":    {1: (0, 0.05), 2: (0.05, 0.25), 3: (0.25, 0.5),  4: (0.5, 1)},
    "salt":    {1: (0, 0.15), 2: (0.15, 0.25), 3: (0.25, 0.7),  4: (0.7, 1)},
    "piquant": {1: (0, 0.15), 2: (0.15, 0.3),  3: (0.3, 0.6),   4: (0.6, 1)},
    "fat":     {1: (0, 0.25), 2: (0.25, 0.5),  3: (0.5, 0.7),   4: (0.7, 1)},
    "bitter":  {1: (0, 0.2),  2: (0.2, 0.37),  3: (0.37, 0.6),  4: (0.6, 1)},
}

## Pairing rules

In [ ]:
def weight_rule(df, food_weight):
    """The wine should have at least the same body as the food."""
    return df.loc[(df["weight"] >= food_weight[1] - 1) & (df["weight"] <= food_weight[1])]


def acidity_rule(df, food_nonaromas):
    """The wine should be at least as acidic as the food."""
    return df.loc[df["acid"] >= food_nonaromas["acid"][1]]


def sweetness_rule(df, food_nonaromas):
    """The wine should be at least as sweet as the food."""
    return df.loc[df["sweet"] >= food_nonaromas["sweet"][1]]


def bitterness_rule(df, food_nonaromas):
    """Bitter wines do not pair well with bitter foods."""
    if food_nonaromas["bitter"][1] == 4:
        return df.loc[df["bitter"] <= 2]
    return df


def bitter_salt_rule(df, food_nonaromas):
    """Bitter and salt do not go well together."""
    if food_nonaromas["bitter"][1] == 4:
        df = df.loc[df["salt"] <= 2]
    if food_nonaromas["salt"][1] == 4:
        df = df.loc[df["bitter"] <= 2]
    return df


def acid_bitter_rule(df, food_nonaromas):
    """Acid and bitterness do not go well together."""
    if food_nonaromas["acid"][1] == 4:
        df = df.loc[df["bitter"] <= 2]
    if food_nonaromas["bitter"][1] == 4:
        df = df.loc[df["acid"] <= 2]
    return df


def acid_piquant_rule(df, food_nonaromas):
    """Acid and piquant do not go well together."""
    if food_nonaromas["acid"][1] == 4:
        df = df.loc[df["piquant"] <= 2]
    if food_nonaromas["piquant"][1] == 4:
        df = df.loc[df["acid"] <= 2]
    return df


def nonaroma_rules(wine_df, food_nonaromas, food_weight):
    """Apply all non-aroma pairing rules, keeping at least 5 wines per rule."""
    df = weight_rule(wine_df, food_weight)
    rules = [acidity_rule, sweetness_rule, bitterness_rule, bitter_salt_rule, acid_bitter_rule, acid_piquant_rule]
    for rule in rules:
        df_test = rule(df, food_nonaromas)
        if df_test.shape[0] > 5:
            df = df_test
    return df

## Congruent and contrasting pairings

In [ ]:
def sweet_pairing(df, food_nonaromas):
    if food_nonaromas["sweet"][1] == 4:
        df["pairing_type"] = np.where(
            (df.bitter == 4) | (df.fat == 4) | (df.piquant == 4) | (df.salt == 4) | (df.acid == 4),
            "contrasting", df.pairing_type,
        )
    return df


def acid_pairing(df, food_nonaromas):
    if food_nonaromas["acid"][1] == 4:
        df["pairing_type"] = np.where(
            (df.sweet == 4) | (df.fat == 4) | (df.salt == 4),
            "contrasting", df.pairing_type,
        )
    return df


def salt_pairing(df, food_nonaromas):
    if food_nonaromas["salt"][1] == 4:
        df["pairing_type"] = np.where(
            (df.bitter == 4) | (df.sweet == 4) | (df.piquant == 4) | (df.fat == 4) | (df.acid == 4),
            "contrasting", df.pairing_type,
        )
    return df


def piquant_pairing(df, food_nonaromas):
    if food_nonaromas["piquant"][1] == 4:
        df["pairing_type"] = np.where(
            (df.sweet == 4) | (df.fat == 4) | (df.salt == 4),
            "contrasting", df.pairing_type,
        )
    return df


def fat_pairing(df, food_nonaromas):
    if food_nonaromas["fat"][1] == 4:
        df["pairing_type"] = np.where(
            (df.bitter == 4) | (df.sweet == 4) | (df.piquant == 4) | (df.salt == 4) | (df.acid == 4),
            "contrasting", df.pairing_type,
        )
    return df


def bitter_pairing(df, food_nonaromas):
    if food_nonaromas["bitter"][1] == 4:
        df["pairing_type"] = np.where(
            (df.sweet == 4) | (df.fat == 4) | (df.salt == 4),
            "contrasting", df.pairing_type,
        )
    return df


def congruent_pairing(pairing_type, max_food_nonaroma_val, wine_nonaroma_val):
    if pairing_type == "congruent":
        return "congruent"
    elif wine_nonaroma_val >= max_food_nonaroma_val:
        return "congruent"
    return ""


def congruent_or_contrasting(df, food_nonaromas):
    """Classify each wine-food pair as congruent, contrasting, or neither."""
    max_nonaroma_val = max(v[1] for v in food_nonaromas.values())
    most_defining_tastes = [k for k, v in food_nonaromas.items() if v[1] == max_nonaroma_val]

    df["pairing_type"] = ""
    for m in most_defining_tastes:
        df["pairing_type"] = df.apply(
            lambda x: congruent_pairing(x["pairing_type"], food_nonaromas[m][1], x[m]), axis=1
        )

    for rule in [sweet_pairing, acid_pairing, salt_pairing, piquant_pairing, fat_pairing, bitter_pairing]:
        df = rule(df, food_nonaromas)
    return df